In [1]:
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pandas as pd 
import numpy as np 

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(
    log_dir="runs/LSTM"
)


from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import os

print(os.getcwd(), device)


/home/eshaan/ML-CODE cuda


[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
from Word2Vec.main import simpleW2V, encoder, most_similar, embedded_sentence, get_weight_vector

encoder = encoder('/home/eshaan/ML-CODE/datasets/hp_books',5000)
word_idx_dict, encoded_chunks, idx_word_dict = encoder.chunk_encoder(50 , True, False)



[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...


[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
print(encoded_chunks[-1])

['i', 'he', 'will', 'the', 'had', 'not', 'for', 'all', 'was', 'and', 'the', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']


In [ ]:

vocab_size = 5000
feature_count = 50

checkpoint = torch.load(r'/home/eshaan/ML-CODE/Word2Vec/checkpoint1.pth')

w2v_model = simpleW2V(vocab_size,feature_count)
w2v_model.load_state_dict(checkpoint['model'])
w2v_model.state_dict()

embedding_w = w2v_model.embedding.weight

# most_similar(w2v_model, "harry", word_idx_dict,"out",20)

In [ ]:
class simpleLSTM(nn.Module):

    def __init__(self, embedding_w, vocab_size, embedding_dim, context_dim, device):
        super().__init__()
            
        if not device:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device

        self.context_dim = context_dim

        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim=embedding_dim
        )
        self.embedding.weight.data.copy_(embedding_w)

        self.F_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + context_dim,
                out_features = context_dim
            ),
            nn.Sigmoid()
        )

        self.I_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + context_dim,
                out_features = context_dim
            ),
            nn.Sigmoid()
        )

        self.C_bar_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + context_dim,
                out_features = context_dim
            ),
            nn.Tanh()
        )

        self.O_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + context_dim,
                out_features = context_dim
            ),
            nn.Sigmoid()
        )

        self.output = nn.Linear(
            in_features= context_dim,
            out_features= vocab_size
        )
        
    def forward(self, encoded_input_sequence):
        # input_dim = (batch, Timesteps)

        batch_size,timesteps = encoded_input_sequence.shape
        # if not LTM_context :
        LTM_context = torch.zeros(size=(batch_size,self.context_dim), device=self.device)
        # if not STM_context :
        STM_context = torch.zeros(size=(batch_size,self.context_dim), device=self.device)


        embedded_input_sequence = self.embedding(encoded_input_sequence) #shape = (batch, Timesteps, embedding_dim)
        logits = []
        for timestep in range(timesteps) : 
            token = embedded_input_sequence[: ,timestep,:]
            F_t = self.F_t(torch.cat([STM_context, token], dim=1))
            LTM_context = F_t * LTM_context

            I_t = self.I_t(torch.cat([STM_context, token], dim=1))
            C_bar_t = self.C_bar_t(torch.cat([STM_context, token], dim=1))
            LTM_context = LTM_context + ( I_t * C_bar_t)

            O_t = self.O_t(torch.cat([STM_context, token], dim=1))
            STM_context = torch.tanh(LTM_context) * O_t
            logits.append(self.output(STM_context))
        logits = torch.stack(logits, dim=1)
        return logits


input                       (B, T)

embedding
                            (B, T, E)

for timestep t:

token                       (B, E)
STM_context                 (B, H)
        ↓ concatenate
combined                    (B, E+H)

F_t                         (B, H)
I_t                         (B, H)
C_bar_t                     (B, H)
O_t                         (B, H)

LTM_context                 (B, H)
STM_context                 (B, H)

Linear(H → V)
logits_t                    (B, V)

after all T timesteps:

logits                      (B, T, V)

In [ ]:
def lstm_dataset(Dataset):

    def __init__(self, encoded_sentences):
        self.samples = []

        X = encoded_sentences[:-1]
        y = encoded_sentences[1:]

        self.samples.append(X, y)

    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        return self.samples[idx][0], self.samples[idx][1]

def lstm_dataloader(batch_size, encoded_sentences) :
    dataset = lstm_dataset(encoded_sentences)
    train_loader = DataLoader(dataset, 
                              batch_size,
                              pin_memory=True,
                              shuffle=True)

    return train_loader

In [ ]:
def training_loop(model, epochs, train_loader, optimizer, criterion):

    for epoch in range(epochs):

        for context,target in train_loader:

            embedded_context = model.embedding(context)